# 07 · Summary statistics

From the $\tau$ of several lines, `DeltaFieldBuilder` forms, per skewer, three kinds of overflux (review notes, Sec.
Overflux):

- the overflux of every line, $\delta_j = F_j/\langle F_j\rangle - 1$;
- the products $\delta_A$ of every subset $A$ of lines;
- the total, $\delta_{\rm tot} = \prod_j F_j/\langle\prod_j F_j\rangle - 1$, which decomposes **exactly** into the subset
  terms.

`OverfluxStatistics1D` gives P1D and $\xi_{\rm 1D}$ of the total, of every line pair and of every order
$\lvert A\rvert + \lvert B\rvert$. P1D and $\xi_{\rm 1D}$ are an exact discrete Fourier pair, with
$P_{ab} = (dv/N)\, \tilde F_a \tilde F_b^*$ (review notes). `OverfluxStatistics3D` does the same on a grid of
skewers.

§3 tests the whole chain on a Gaussian random field whose spectrum is known. Runtime: ~2 min.

**Requires:** the optical depth of several lines (06) and their offsets $\Delta v_j$ (01).  
**Independent of:** how the $\tau$ was computed (core or simulation).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'samplers', 'fields', 'tau', 'overflux', 'stats',)); plt.show()

## 1 · The flux of two lines on one axis

The toy box of notebook 00: 50 cMpc at $z = 2.2$, with H I and Si III densities $n = \bar n (1 + \delta)$ from one
`PowerLawGRF` realisation. The skewers cross the box once. $\tau$ is integrated with wrapped detunings on a grid of
exactly one period, and both lines are put on the Ly$\alpha$ frame with `wrap_to_ref_line` (notebook 06 explains both
steps). From here on only $\tau$ and $F = e^{-\tau}$ per line on that common axis are needed.

**The overflux of two lines of the same gas.** A one-dimensional example makes the decomposition explicit. Line 2 is a
shifted, weaker copy of line 1, $\tau_2(v) = \tfrac12 \tau_1(v - \Delta v)$, with $\Delta v = 100$ km/s and
$\tau_1 = e^{\delta}$ for a 1D GRF $\delta$ ($\sigma_\delta = 1$). `DeltaFieldBuilder(tau, labels)` returns:

- the overflux of every line;
- the subset products;
- the total;
- the normalisation term $C$ of every skewer.

In [ ]:
from skewerjax.core import PowerLawGRF, DeltaFieldBuilder
from skewerjax.core.stats import OverfluxStatistics1D, OverfluxStatistics3D
from skewerjax.utils.plotting import plot_orders_superimposed, plot_pk_recovery, hybrid_k_edges, logbin
N_sk, N_v, Lv = 400, 1000, 1000.0                         # skewers, pixels, velocity length [km/s]
dv, shift = Lv / N_v, 100                                  # the second line sits 100 km/s away
grf = PowerLawGRF(1.0, 2 * np.pi / 100, 0.5, L=Lv, N=N_v)
grf = PowerLawGRF(1.0 / float(grf.variance()), 2 * np.pi / 100, 0.5, L=Lv, N=N_v)     # sigma_delta = 1
delta = jax.vmap(grf.sample)(jax.random.split(jax.random.PRNGKey(7), N_sk))
tau = jnp.stack([jnp.exp(delta), 0.5 * jnp.exp(jnp.roll(delta, shift, axis=-1))])        # line 2: same gas, shifted, weaker
ov = DeltaFieldBuilder(tau, labels=["1", "2"], keep_fields=True).build()
print("subset keys:", ov.subset_keys, "| C_s of the first skewers:", np.round(np.asarray(ov.C_per_sample[:3, 0]), 4))
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True, layout="constrained")
vv = np.arange(N_v) * dv
for j, lab in enumerate(ov.labels):
    axs[0].plot(vv, ov.fields[j, 0], color=list(TOL.values())[j], label=f"$F_{lab}$"); axs[1].plot(vv, ov.delta_terms[lab][0], color=list(TOL.values())[j])
axs[1].plot(vv, ov.total[0], color=FG, lw=1.2, label=r"$\delta_{\rm tot}$")
axs[0].set(ylabel="F", xlim=(0, 300)); axs[1].set(xlabel="v [km/s]", ylabel=r"$\delta$"); axs[0].legend(); axs[1].legend()
add_panel_letters(axs); fig.suptitle("Two lines of the same gas: the second is a shifted, weaker copy", fontsize=22); plt.show()

## 2 · P1D and $\xi_{\rm 1D}$ by order

Order $n$ collects the terms $(A, B)$ with $\lvert A\rvert + \lvert B\rvert = n$. Order 0 is the normalisation term
$\mathcal{C}$. The orders add up to the total to machine precision.

In [ ]:
st = OverfluxStatistics1D(ov, dv).compute_p1d().compute_xi1d()
fig, axs = plot_orders_superimposed(st, "P", delta_v_list=[0.0, shift * dv], figsize=(13, 9))
axs[0].legend(fontsize=15, ncol=2, loc="lower left")
fig.suptitle(r"P1D: orders 2, 3, 4 add up to the total; the cross terms peak at $k = 2\pi n/\Delta v$ (dashed)", fontsize=20)
fig, axs = plot_orders_superimposed(st, "xi", delta_v_list=[0.0, shift * dv], max_lag=300.0, figsize=(12, 8))
fig.suptitle(r"$\xi_{\rm 1D}$: the cross term peaks at the line separation (dashed)", fontsize=20); plt.show()

**The terms of two lines.** With two lines,

$$
F = \bar F_1 \bar F_2\, (1 + \delta_1)(1 + \delta_2),\qquad
\delta = \frac{\delta_1 + \delta_2 + \delta_1\delta_2 - C_{12}}{1 + C_{12}},\qquad C_{12} = \langle \delta_1 \delta_2 \rangle,
$$

where $C_{12}$ is the normalisation term ($\mathcal{C}$ below). Every term of $\xi_{\rm 1D}$ (real space) or P1D (Fourier
space) is a pair of subsets $A, B$ of $\{1, 2\}$, of order $\lvert A\rvert + \lvert B\rvert$:

| $A \backslash B$ | $\mathcal{C}$ (order 0) | 1 (order 1) | 2 (order 1) | 12 (order 2) |
|---|---|---|---|---|
| $\mathcal{C}$ | $+\mathcal{C}\mathcal{C}$ | 0 | 0 | $-C_{12}\mathcal{C}$ |
| 1 | 0 | $\xi_{11}$, $P_{11}$ | $\xi_{21}$, $P_{21}$ | $\xi_{(12)1}$, $P_{(12)1}$ |
| 2 | 0 | $\xi_{12}$, $P_{12}$ | $\xi_{22}$, $P_{22}$ | $\xi_{(12)2}$, $P_{(12)2}$ |
| 12 | $-\mathcal{C} C_{12}$ | $\xi_{1(12)}$, $P_{1(12)}$ | $\xi_{2(12)}$, $P_{2(12)}$ | $\xi_{(12)(12)}$, $P_{(12)(12)}$ |

After the ensemble average,

$$
\xi = \frac{-C_{12}\mathcal{C} + \xi_{11} + \xi_{22} + \xi_{12,{\rm sym}} + \xi_{1(12),{\rm sym}} + \xi_{2(12),{\rm sym}} + \xi_{(12)(12)}}{(1 + C_{12})^2},
$$

$$
P(k) = \frac{P_{11} + P_{22} + P_{12,{\rm sym}} + P_{1(12),{\rm sym}} + P_{2(12),{\rm sym}} + P_{(12)(12)} - 2\pi C_{12}^2\, \delta_D(k)}{(1 + C_{12})^2},
$$

where "sym" adds the pair and its transpose: 1/2 with 2/1, 1/12 with 12/1, and 2/12 with 12/2. In Fourier space the
constant pieces only touch the zero mode.

For a second line that is a shifted copy of the first, $\delta_2(v) \propto \delta_1(v - \Delta v)$, the cross term is
$P_{12,{\rm sym}} \propto 2\cos(k\Delta v)\, P_{11}$. It oscillates with period $2\pi/\Delta v$ in $k$ (the dashed
lines above): the Fourier image of the $\xi_{\rm 1D}$ peak at $\Delta v$.

### Each order, term by term

The same decomposition one level down. Every term $(A, B)$ of order $\lvert A\rvert + \lvert B\rvert = n$ is shown,
with $(A, B)$ and $(B, A)$ drawn together as A–B (their sum, which is real). $\mathcal{C}$ is the order-0 normalisation
term.

- Order 2 holds the Gaussian-like products (1–1, 1–2, 2–2) and the normalisation cross term $\mathcal{C}$–12.
- Orders 3 and 4 are the non-linear corrections of the product of the two fluxes.

In [ ]:
from skewerjax.utils.plotting import plot_terms_by_order
fig, axs = plot_terms_by_order(st, "P")
fig.suptitle("P1D: every order split into its terms; the terms add up to the order", fontsize=20); plt.show()
fig, axs = plot_terms_by_order(st, "xi", max_lag=300.0)
for ax in axs:
    ax.axvline(shift * dv, color=FG, ls=":", lw=1.2)
fig.suptitle(r"$\xi_{\rm 1D}$ by order and term: the cross terms 1–2 carry the line separation (dotted)", fontsize=20); plt.show()
P_terms = {n: sum(np.asarray(st.Pk_total_from_terms_contributions[p]) for p in st.term_pairs if st.pair_orders[p] == n) for n in st.Pk_total_by_order}
print("max |sum of the terms - order| / max|total| per order:",
      {n: f"{np.max(np.abs(P_terms[n] - np.asarray(st.Pk_total_by_order[n]))) / np.max(np.abs(np.asarray(st.Pk_total))):.1e}" for n in P_terms})

## 3 · Recovering a known spectrum: a systematic study

One `PowerLawGRF` with input spectrum $P_{\rm in}(k) = A_s (k/k_0)^{n_s - 1}$ sets the H I and Si III densities of a
50 cMpc box ($128^3$ nodes, $z = 2.2$, $T = 1.5\times10^4$ K, no peculiar velocities):

$$
n_{\rm HI} = \bar n_{\rm HI}\,(1 + \delta),\qquad n_{\rm SiIII} = \bar n_{\rm SiIII}\,(1 + \delta) .
$$

No transform or threshold is applied. $A_s$ keeps $1 + \delta > 0$ in the realisation, and the minimum is printed
below. Every signal is measured with the same estimator and compared with its expectation, in four steps:

- (a) 3D fields;
- (b) 1D overdensities along skewers;
- (c) 1D overfluxes;
- (d) 3D statistics of the flux.

**The construction.** Both ions use the **same** realisation $\delta$. Si III traces the same gas as H I, which is what
correlates the two lines in the forest (the cross terms of §2). An independent realisation is drawn only in (a), to
check the other analytic case.

The skewers run along $y$ through every fourth grid column ($32 \times 32$ skewers), with 8192 points each; $\tau$
uses wrapped detunings. The grid is fine enough ($\Delta = 0.39$ cMpc, about 49 km/s) for the thermal widths to show
in (c).

In [ ]:
from types import SimpleNamespace
from skewerjax.utils import tutorial
from skewerjax.utils.plotting import binned_power, plot_p1d_input_vs_measured
N3, L3, step = 128, 50.0, 4                                                     # nodes per axis, box [cMpc], skewer every step-th column
A_s, k_0, n_s = 0.018, 2.0, -1.25                                                       # P(k) = A_s (k / k_0)^(n_s - 1)
grf3 = PowerLawGRF(A_s=A_s, k_0=k_0, n_s=n_s, L=(L3,) * 3, N=(N3,) * 3)
n_bar = {"HI": 2.5e-12, "SiIII": 2.5e-12 / 50}
d_HI = GRFDensity(jax.random.PRNGKey(11), grf3, n_bar=n_bar["HI"])                     # one realisation for both ions
dens3 = {"HI": d_HI, "SiIII": GRFDensity(None, grf3, n_bar=n_bar["SiIII"], delta_grid=d_HI.delta_grid)}
d_ind = GRFDensity(jax.random.PRNGKey(12), grf3, n_bar=n_bar["SiIII"])                 # an independent one, for (a) only
print(f"sigma_delta = {float(jnp.sqrt(grf3.variance())):.3f}; min(1 + delta): shared realisation {float(d_HI.min_one_plus_delta):.3f}, "
      f"independent one {float(d_ind.min_one_plus_delta):.3f}")
# the densities at every node of the grid (events (a, x, y, z)), the line of sight (y) last: arrays (x, z, y)
gx = np.arange(N3) * (L3 / N3)
X, Z, Y = np.meshgrid(gx, gx, gx, indexing="ij")
ev3 = jnp.asarray(np.stack([np.ones(X.size), X.ravel(), Y.ravel(), Z.ravel()], -1))
n3 = {k: np.asarray(d(ev3)).reshape(N3, N3, N3) for k, d in (("HI", d_HI), ("SiIII", dens3["SiIII"]), ("SiIII_ind", d_ind))}
dn = lambda n: n / n.mean() - 1.0
# the skewers: every step-th grid column (x_i, z_j), along y, 8192 points; tau of both lines, periodic, on the Lya frame
Ns, n_sub = N3 // step, 8192 // N3
ys = np.arange(N3 * n_sub) * (L3 / (N3 * n_sub))
P3 = np.stack(np.meshgrid(gx[::step], gx[::step], indexing="ij"), -1).reshape(-1, 2)
pts3 = jnp.asarray(np.stack([np.repeat(P3[:, :1], ys.size, 1), np.broadcast_to(ys, (P3.shape[0], ys.size)), np.repeat(P3[:, 1:], ys.size, 1)], -1))
sb3d = SkewerBundle(spatial_points=pts3, cosmologies=eds, a0=1 / (1 + z), x0=L3)
sp3 = lambda d: {"temperature": ScalarField(LogRamp(1.5e4, 1.5e4, L3)), "velocity": VectorField(VelocityRamp(0.0, 0.0, L3)),
                 "number_density": ScalarField(d), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
toy3 = Fields({sp: sp3(d) for sp, d in dens3.items()})
a3d = np.asarray(sb3d.scale_factors[0])
P3v = float(phys.c_kms * np.log(a3d.max() / a3d.min())) * ys.size / (ys.size - 1)     # the box in velocity [km/s]
N_v3 = N3 * 4
v3 = (np.arange(N_v3) + 0.5) * P3v / N_v3 + float(phys.c_kms * np.log((1 / (1 + z)) / a3d.max()))
tau3 = np.asarray(wrap_to_ref_line(compute_optical_depth(sb3d, v3, lines2, toy3, a_dagger=1 / (1 + z), velocity_period=P3v,
                                                         resolution_tolerance=0.3), v3, lines2))
F3 = np.exp(-tau3)
u3 = P3v / L3                                                                          # km/s per cMpc (Hubble flow of the box)
print(f"{P3.shape[0]} skewers x {ys.size} points; box {P3v:.0f} km/s ({u3:.1f} km/s per cMpc); <F>: Lya {F3[0].mean():.3f}, "
      f"Si III {F3[1].mean():.4f}, total {F3.prod(0).mean():.3f}")

### (a) 3D fields: exact expectations

At the nodes, $n_i / \langle n_i\rangle - 1 = \delta$ exactly, because $\delta$ has no $k = 0$ mode. So the overdensity of
each ion **is** the realisation. Its isotropic $P(k)$ must recover $P_{\rm in}$ within the scatter of one realisation,
$P/\sqrt{N_{\rm modes}}$ per bin.

**The total density** $n_{\rm tot} = n_{\rm HI} + n_{\rm SiIII}$ has $\langle n_{\rm tot}\rangle = \bar n_{\rm HI} + \bar n_{\rm SiIII}$,
so with the weights $w_i = \bar n_i / \bar n_{\rm tot}$

$$
\delta_{\rm tot} = \frac{n_{\rm tot}}{\langle n_{\rm tot}\rangle} - 1 = w_{\rm HI}\,\delta_{\rm HI} + w_{\rm SiIII}\,\delta_{\rm SiIII},
\qquad w_{\rm HI} + w_{\rm SiIII} = 1 .
$$

- **Same realisation**, $\delta_{\rm HI} = \delta_{\rm SiIII} = \delta$: then $\delta_{\rm tot} = \delta$ exactly, and
  $P_{\rm tot} = P_{\rm in}$.
- **Independent realisations**: the cross spectrum vanishes on average, so
  $P_{\rm tot} = (w_{\rm HI}^2 + w_{\rm SiIII}^2)\, P_{\rm in}$. Here $w_{\rm SiIII} = 1/51$, so the factor is 0.962.

Each curve below is divided by its expectation, so all of them must lie on the input. With $128^3$ nodes the bins
hold up to $10^5$ modes, so the input is averaged over the modes of each bin as well (`tutorial.binned_input_power`).
Otherwise the curvature of $P_{\rm in}$ inside a bin would exceed the error bars.

In [ ]:
k_edges = np.geomspace(2 * np.pi / L3 * 0.999, np.pi * N3 / L3, 16)
kin = lambda kb: np.asarray(grf3.power(kb))
w = {sp: n_bar[sp] / (n_bar["HI"] + n_bar["SiIII"]) for sp in n_bar}
rows = {}
for lab, f, exp_, c, sty in (("δ H I", dn(n3["HI"]), 1.0, EC["HI"], dict(marker="o", ms=7, lw=2)),
                             ("δ Si III", dn(n3["SiIII"]), 1.0, EC["SiIII"], dict(marker="s", ms=6, lw=2, ls="--")),
                             ("δ_tot, same realisation", dn(n3["HI"] + n3["SiIII"]), 1.0, TOL["green"], dict(lw=2.4, ls=":")),
                             ("δ_tot, independent realisations", dn(n3["HI"] + n3["SiIII_ind"]), w["HI"] ** 2 + w["SiIII"] ** 2, TOL["purple"],
                              dict(marker="D", ms=6, lw=1.8, ls="-."))):
    kb, Pb, eb, nm = binned_power(f[None], L3, k_edges)
    rows[lab] = (Pb / exp_, eb / exp_, c, sty, exp_)
Pin_b = tutorial.binned_input_power(grf3, k_edges)                                     # the input averaged over each bin's modes
fig, axs = plt.subplots(2, 1, figsize=(15, 11), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.3]}, layout="constrained")
plot_p1d_input_vs_measured(kb, {k: v[:4] for k, v in rows.items()}, Pin_b, ref_label="input (bin-averaged)", axs=axs)
axs[0].set_ylabel(r"$k P(k)/\pi$ [cMpc$^{-2}$]"); axs[1].set(xlabel=r"$|k|$ [cMpc$^{-1}$]", ylim=(0.7, 1.3))
add_panel_letters(axs); fig.suptitle("(a) 3D: every overdensity on its analytic expectation (the independent sum is divided by w₁² + w₂²)", fontsize=20)
plt.show()
for lab, (Pb, eb, c, sty, exp_) in rows.items():
    print(f"{lab:32s} expectation {exp_:.4f} x P_in: chi2/dof = {np.mean(((Pb - Pin_b) / eb) ** 2):.2f} over {kb.size} bins")
print(f"same realisation: max |delta_tot - delta_HI| = {np.abs(dn(n3['HI'] + n3['SiIII']) - dn(n3['HI'])).max():.1e} (exact)")

### (b) 1D overdensities along the skewers

Along a line of the grid, the 1D spectrum of the overdensity is the projection of the 3D one onto the line of sight.
On a periodic grid this projection is a discrete sum over the transverse modes of the grid, not an integral:

$$
P_{\rm 1D}(k_\parallel) = \frac{1}{L_\perp^2} \sum_{\mathbf{k}_\perp} P_{\rm in}\left(\sqrt{k_\parallel^2 + k_\perp^2}\right)
$$

(`plotting.grf_p1d_along_axis`). This is the exact expectation **at the nodes**.

**Along the skewers**, the 8192 points lie between the nodes, and `GRFDensity` interpolates linearly. Sampling between
nodes multiplies the node spectrum, which is periodic in $k_\parallel$ with period $2\pi/\Delta$, by the window of linear
interpolation:

$$
P_{\rm 1D}^{\rm interp}(k_\parallel) = \mathrm{sinc}^4\!\left(\frac{k_\parallel \Delta}{2}\right) P_{\rm 1D}\big(k_\parallel\ \mathrm{folded\ into}\ [-\pi/\Delta, \pi/\Delta)\big),
\qquad \Delta = L/128
$$

(`tutorial.p1d_expected`). Beyond the grid's Nyquist frequency (dotted) the interpolated field still has power: the
images of the node spectrum, damped by the window. With both effects included, the agreement holds up to high $k$.

The error bars are the exact scatter of this average for one realisation. The skewers share the 3D modes, and every
coarse transverse mode of the $32^2$ skewer grid is the sum of 16 aliased modes of the $128^2$ grid. The scatter
between skewers would therefore underestimate the error.

The total density uses the same realisation, so its expectation is the same.

In [ ]:
one = lambda d: SimpleNamespace(subset_keys=["a"], delta_terms={"a": jnp.asarray(d)}, C_per_sample=jnp.zeros((d.shape[0], 1)), total=jnp.asarray(d))
p1d = lambda d, dx: OverfluxStatistics1D(one(d), dx).compute_p1d()
take = lambda st: (np.asarray(st.k), np.asarray(st.Pk_total).real, np.asarray(st.Pk_total_err).real)
dx3 = L3 / N3
# at the grid nodes: the 32 x 32 skewer columns, 128 nodes each
cols = lambda n: n[::step, ::step].reshape(Ns * Ns, N3)
k_n, P_HI_n, e_HI_n = take(p1d(dn(cols(n3["HI"])), dx3))
_, P_tot_n, e_tot_n = take(p1d(dn(cols(n3["HI"] + n3["SiIII"])), dx3))
# along the skewers themselves: 8192 points per skewer, linear interpolation between the nodes
n_sk = {sp: np.asarray(d(sb3d.quadri_vectors[0])) for sp, d in dens3.items()}
dys = ys[1] - ys[0]
k_i, P_HI_i, e_HI_i = take(p1d(dn(n_sk["HI"]), dys))
_, P_Si_i, e_Si_i = take(p1d(dn(n_sk["SiIII"]), dys))
_, P_tot_i, e_tot_i = take(p1d(dn(n_sk["HI"] + n_sk["SiIII"]), dys))
sel_n = slice(1, N3 // 2 + 1)
sel_i = (k_i > 0) & (k_i <= 3 * np.pi / dx3) & (tutorial.linear_interp_window(k_i, dx3) > 1e-3)   # not at the window's zeros
E_n, S_n = tutorial.p1d_expected(grf3, k_n[sel_n], axis=1, return_sigma=True, step=step)
E_i, S_i = tutorial.p1d_expected(grf3, k_i[sel_i], axis=1, interpolated=True, return_sigma=True, step=step)
# errors: the exact scatter of the mean P1D over the 32 x 32 skewer columns for one realisation (the skewers share the
# 3D modes, so the scatter between skewers underestimates it)
e_HI_n = e_tot_n = np.zeros_like(P_HI_n); e_HI_n[sel_n] = S_n; e_tot_n = e_HI_n
e_HI_i = np.zeros_like(P_HI_i); e_HI_i[sel_i] = S_i; e_Si_i = e_tot_i = e_HI_i
fig, axs = plt.subplots(2, 2, figsize=(24, 11), sharex="col", gridspec_kw={"height_ratios": [2.2, 1.3]}, layout="constrained")
plot_p1d_input_vs_measured(k_n[sel_n], {"δ H I": (P_HI_n[sel_n], e_HI_n[sel_n], EC["HI"], dict(marker="o", ms=6, lw=2)),
                                         "δ_tot": (P_tot_n[sel_n], e_tot_n[sel_n], TOL["green"], dict(lw=2.4, ls=":"))},
                           E_n, ref_label="discrete projection of the input", axs=axs[:, 0])
plot_p1d_input_vs_measured(k_i[sel_i], {"δ H I": (P_HI_i[sel_i], e_HI_i[sel_i], EC["HI"], dict(lw=2)),
                                         "δ Si III": (P_Si_i[sel_i], e_Si_i[sel_i], EC["SiIII"], dict(lw=2, ls="--")),
                                         "δ_tot": (P_tot_i[sel_i], e_tot_i[sel_i], TOL["green"], dict(lw=2.4, ls=":"))},
                           E_i, ref_label="projection × interpolation window", axs=axs[:, 1])
for ax in axs[:, 1]:
    ax.axvline(np.pi / dx3, color=DIM, ls=":", lw=1.4)
axs[0, 0].set_title("(b1) at the grid nodes (128 per skewer)", fontsize=18); axs[0, 1].set_title("(b2) along the skewers (8192 points)", fontsize=18)
for a in axs[1]:
    a.set(xlabel=r"$k_\parallel$ [cMpc$^{-1}$]", ylim=(0.8, 1.2))
add_panel_letters(axs.ravel()); fig.suptitle("(b) 1D overdensities along the skewers against the projected input", fontsize=21); plt.show()
chi = lambda P_, e_, E_: np.mean(((P_ - E_) / e_) ** 2)
print(f"nodes: delta_HI chi2/dof {chi(P_HI_n[sel_n], e_HI_n[sel_n], E_n):.2f}; along the skewers: delta_HI {chi(P_HI_i[sel_i], e_HI_i[sel_i], E_i):.2f}, "
      f"delta_SiIII {chi(P_Si_i[sel_i], e_Si_i[sel_i], E_i):.2f}, delta_tot {chi(P_tot_i[sel_i], e_tot_i[sel_i], E_i):.2f} "
      f"(k up to 3 x the grid Nyquist, the zeros of the window excluded; errors: one realisation, exact)")
print(f"max |measured / expected - 1|: nodes {np.abs(P_HI_n[sel_n] / E_n - 1).max():.3f}, along the skewers {np.abs(P_HI_i[sel_i] / E_i - 1).max():.3f}")

### (c) 1D overfluxes: what is not expected to match, and why

The overflux $\delta_F = F/\langle F\rangle - 1$ is not the density. Four effects separate its P1D from the projected
input, and none of them is a numerical error:

- **The non-linear $F(\tau)$.** $F = e^{-\tau}$, so even for $\tau \propto n$ the overflux is
  $\delta_F \approx -\bar\tau\, \delta + \mathcal{O}(\delta^2)$. Its large-scale amplitude is a bias
  $b_F^2 \approx \bar\tau^2$, measured below at the lowest $k$. The second-order terms add a small, nearly white
  contribution.
- **Thermal broadening.** Every element absorbs over a Gaussian of Doppler parameter $b$. At linear order this
  multiplies the spectrum by $e^{-k^2 b^2/2}$ ($k$ in s/km), with $b$ = 15.7 km/s for H I and 3.0 km/s for Si III at
  $1.5\times10^4$ K. The H I overflux therefore falls far below the input at high $k$; Si III much less.
- **Peculiar velocities** move the absorption in redshift space (Kaiser boost on large scales, damping on small ones).
  This toy has none, so they play no role here; in a simulation they do.
- **The shift onto the Ly$\alpha$ frame.** `wrap_to_ref_line` moves Si III by $\Delta v$, which is not a whole number of
  pixels. It interpolates linearly between pixels: a shift by a fraction $f$ of a pixel $dv$ multiplies the power by
  $1 - 2f(1-f)\,[1 - \cos(k\, dv)]$. The 12.3 km/s pixels used here make this visible at the highest $k$ (it is the
  "pixel-shift window" below). Ly$\alpha$ is the reference and is not shifted.
- **The line offset.** The total flux $F_{{\rm Ly}\alpha} F_{\rm SiIII}$ (equivalently, the sum of the $\tau$) contains the
  Ly$\alpha \times$ Si III cross term $2\cos(k\Delta v) P_{12}$ of §2. It modulates the total with period
  $2\pi/\Delta v$ in $k$ (dotted lines; $\Delta v$ modulo the box period).

Below, every overflux is divided by its own $b_F^2$, and the reference is the linear expectation without the thermal
kernel: the projected input with its interpolation window, converted to velocity through the Hubble flow of the box.
The residual panel overlays the thermal kernels, which is where the measured ratios should fall. The band ends at the
Nyquist frequency of the density grid.

In [ ]:
dv3 = P3v / N_v3
fluxes = {"Lyα": F3[0], "Si III": F3[1], "total (F_Lyα F_SiIII)": F3.prod(0)}
b_th = {"Lyα": np.sqrt(2 * phys.kB * 1.5e4 / lines2.mass_amu[0]) / 1e3, "Si III": np.sqrt(2 * phys.kB * 1.5e4 / lines2.mass_amu[1]) / 1e3}
E_lin = lambda k_v, b: tutorial.p1d_expected(grf3, k_v * u3, axis=1, interpolated=True) * u3 * np.exp(-0.5 * (k_v * b) ** 2)
frows, info = {}, {}
for (lab, F), c, sty in zip(fluxes.items(), (EC["HI-LyA"], EC["SiIII-1206"], FG), (dict(lw=2.4), dict(lw=2.0, ls="--"), dict(lw=1.6, ls=":"))):
    kv, Pv, ev = take(p1d(F / F.mean() - 1, dv3))
    s_ = (kv > 0) & (kv * u3 <= np.pi / dx3)                                         # the band of the density grid
    b = b_th.get(lab, b_th["Lyα"])
    bF2 = np.median(Pv[s_][:4] / E_lin(kv[s_][:4], b))                                  # the large-scale bias, b_F^2
    info[lab] = dict(bF2=bF2, tau_bar=-np.log(F.mean()))
    frows[f"{lab}: δ_F / b_F, b_F² = {bF2:.2g}"] = (Pv[s_] / bF2, ev[s_] / bF2, c, sty)
k_v = kv[s_]
fig, axs = plt.subplots(2, 1, figsize=(16, 11), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.3]}, layout="constrained")
plot_p1d_input_vs_measured(k_v, frows, E_lin(k_v, 0.0), ref_label="linear, no thermal kernel", axs=axs)
f_sh = np.mod(abs(float(lines2.delta_v[1])) / dv3, 1.0)                               # Si III's shift in pixels: fractional part
W_shift = 1 - 2 * f_sh * (1 - f_sh) * (1 - np.cos(k_v * dv3))                           # linear interpolation by a fraction f_sh
for lab, c, extra, txt in (("Lyα", EC["HI-LyA"], 1.0, ""), ("Si III", EC["SiIII-1206"], W_shift, r" $	imes$ pixel-shift window")):
    axs[1].plot(k_v, np.exp(-0.5 * (k_v * b_th[lab]) ** 2) * extra, color=c, ls="-.", lw=1.8, label=rf"thermal kernel $e^{{-k^2 b^2/2}}${txt}, {lab}")
axs[1].set(xscale="log", yscale="linear", ylim=(0.4, 1.3), xlabel=r"$k$ [s km$^{-1}$]"); axs[1].legend(fontsize=15, loc="lower left")
for d_ in (1, 2, 3):
    axs[0].axvline(2 * np.pi * d_ / abs(float(lines2.delta_v[1]) % P3v), color=DIM, ls=":", lw=1)
add_panel_letters(axs); fig.suptitle("(c) 1D overfluxes: biased, smoothed by the thermal widths, and coupled by the line offset", fontsize=21); plt.show()
for lab, d in info.items():
    print(f"{lab:24s} b_F^2 = {d['bF2']:.3g}  (linear estimate tau_bar^2 = {d['tau_bar'] ** 2:.3g}, tau_bar = -ln<F> = {d['tau_bar']:.3g})")
for lab, key in (("Lyα", "Lyα"), ("Si III", "Si III")):
    rr = frows[[r for r in frows if r.startswith(lab + ":")][0]][0] / E_lin(k_v, 0.0)
    W_ = W_shift[-1] if key == "Si III" else 1.0
    print(f"{lab}: measured / linear at k = {k_v[-1]:.3f} s/km: {rr[-3:].mean():.2f}; thermal kernel there {np.exp(-0.5 * (k_v[-1] * b_th[key]) ** 2):.2f}"
          + (f", x pixel-shift window {W_:.2f} = {W_ * np.exp(-0.5 * (k_v[-1] * b_th[key]) ** 2):.2f} (shift {abs(float(lines2.delta_v[1])) / dv3:.2f} pixels)" if key == "Si III" else ""))

### (d) The 3D statistics of the flux

The $32 \times 32$ skewers of (b) form a regular transverse grid (row-major, $N_{\rm sk} = N_x N_y$), so
`OverfluxStatistics3D` measures $P_{\rm 3D}(k_\perp, k_\parallel)$ and $\xi_{\rm 3D}(r_\perp, \pi)$ of the
Ly$\alpha$ overflux directly. The line of sight is converted to cMpc through the Hubble flow of the box.

**The expectation is built the same way as in (b) and (c)** (`tutorial.p3d_expected_on_skewer_grid`):

- the skewers sample every fourth column, so the transverse modes of the $128^2$ grid **alias** onto the $32^2$ grid
  of the skewers (16 images per mode);
- along the line of sight: the node spectrum, folded, times the interpolation window;
- the thermal kernel $e^{-k_\parallel^2 b^2/2}$ of H I.

The expectation is binned exactly as the measurement (`bin_p3d_kperp(..., Pk3d_input=...)`). $b_F^2$ is the one
measured in (c), so nothing is fitted here. Only the band of the density grid, $k_\parallel \le \pi/\Delta$, is shown.

**What is not expected to match:**

- the non-linear terms of $F(\tau)$, which add small-scale power;
- the slight non-linearity of the Hubble flow across 50 cMpc, a few per cent in the conversion of $k_\parallel$.

$\xi_{\rm 3D}$ shows the same anisotropy in real space: the correlation is shorter across the line of sight (set by the
density field only) than along it, where the thermal widths add their smoothing.

In [ ]:
from skewerjax.utils.plotting import tol_cmap
dpar = L3 / N_v3                                                                       # the line of sight in cMpc of the box
st3 = OverfluxStatistics3D(DeltaFieldBuilder(jnp.asarray(tau3[:1])).build(), dpar, step * L3 / N3).compute_xi3d()
Pb, kperp, _, kpar, _ = (np.asarray(x) for x in st3.bin_p3d_kperp(5))
b_cMpc = b_th["Lyα"] / u3
E3 = tutorial.p3d_expected_on_skewer_grid(grf3, step, np.asarray(st3.k_par), axis=1) * np.exp(-0.5 * (np.asarray(st3.k_par) * b_cMpc) ** 2)[None, None]
Eb = np.asarray(st3.bin_p3d_kperp(5, Pk3d_input=jnp.asarray(E3))[0])                     # the expectation, binned as the measurement
bF2_3d = info["Lyα"]["bF2"]                                                           # the bias of (c): nothing is fitted here
keep = (kpar > 0) & (kpar <= np.pi / dx3)                                             # the band of the density grid
kpar, Pb, Eb = kpar[keep], Pb[:, keep], Eb[:, keep]
cm = plt.get_cmap(CMAPS["line"])
fig, axs = plt.subplots(2, 2, figsize=(24, 11), layout="constrained", gridspec_kw={"height_ratios": [2.2, 1.3], "width_ratios": [1.2, 1]})
for i in range(len(kperp)):
    c = cm(i / max(len(kperp) - 1, 1))
    axs[0, 0].loglog(kpar, Pb[i] / bF2_3d, color=c, lw=2, label=rf"$k_\perp$ = {kperp[i]:.2f}")
    axs[0, 0].loglog(kpar, Eb[i], color=c, **MS["reference"])
    axs[1, 0].semilogx(kpar, Pb[i] / bF2_3d / Eb[i], color=c, lw=2)
axs[1, 0].axhline(1, color=FG, ls="--", lw=1.2); axs[1, 0].set_ylim(0, 2)
axs[0, 0].set(ylabel=r"$P_{\rm 3D}/b_F^2$ [cMpc$^3$]", title=r"P3D of the Lyα overflux in bins of $k_\perp$ [cMpc$^{-1}$] (light: expectation)")
axs[1, 0].set(xlabel=r"$k_\parallel$ [cMpc$^{-1}$]", ylabel="measured / expected"); axs[0, 0].legend(fontsize=15, ncol=2)
xb, rperp, _, pi, _ = st3.bin_xi3d_rperp(12)
xb = np.asarray(xb); lim = float(np.nanmax(np.abs(xb)))
im = axs[0, 1].pcolormesh(np.asarray(pi), np.asarray(rperp), xb, cmap=tol_cmap("BuRd"), vmin=-lim, vmax=lim, shading="auto")
fig.colorbar(im, ax=axs[0, 1], label=r"$\xi_{\rm 3D}$"); axs[0, 1].set(xlabel=r"$\pi$ [cMpc]", ylabel=r"$r_\perp$ [cMpc]", title=r"$\xi_{\rm 3D}(r_\perp, \pi)$ of the Lyα overflux")
mid = np.argmin(np.abs(np.asarray(pi)))
axs[1, 1].plot(np.asarray(rperp), xb[:, mid], color=EC["HI-LyA"], lw=2, label=r"$\pi \approx 0$ (across)")
axs[1, 1].plot(np.abs(np.asarray(pi)), xb[0], color=TOL["yellow"], lw=2, ls="--", label=r"$r_\perp \approx 0$ (along)")
axs[1, 1].axhline(0, color=FG, lw=0.8); axs[1, 1].set(xlabel="separation [cMpc]", ylabel=r"$\xi_{\rm 3D}$"); axs[1, 1].legend(fontsize=15)
add_panel_letters(axs.ravel()); fig.suptitle("(d) The 3D statistics of the Lyα overflux on the 32 × 32 skewers", fontsize=21); plt.show()
r_ = Pb / bF2_3d / Eb
print(f"b_F^2 from (c) = {bF2_3d:.3g}; measured / expected: median over k_par < 1 cMpc^-1 {np.median(r_[:, kpar < 1.0]):.3f}, "
      f"per k_perp bin at the grid Nyquist k_par = {kpar[-1]:.2f}: {np.round(r_[:, -1], 2)}")

All the statistics kernels (`p1d_kernel`, `p3d_kernel`, `xi_from_power`, `binned_mean`) are jitted. The recovery and
exactness checks are in `scripts/validation/check_summary_statistics.py`.